# 01 · API 基础：调用、追问与停止

按顺序手动运行，完成一个会话后再试可选的双路并行。**不要直接运行全部单元格**：需要在分析期间运行追问，结束时再运行停止。

本示例只显示普通文本。第 3 步上传视频，第 5 步启动线上分析并产生用量；默认 60 秒自动停止。没有自动重试或自动重连。

首次安装：在仓库根目录执行 `python -m pip install -r examples/requirements-basic.txt`，选择 **Python (VLX-Flow Demos)** 内核。此 Notebook 不需要 anywidget。

[Realtime 文档](https://s.apifox.cn/46c7cae8-f2d7-4780-a960-e531d6c5eaa8/9598749m0) · [进阶交互版](02_interactive_playground.ipynb)

## 1 · 导入依赖与定位项目

导入不会连接 API。视频使用仓库内相对路径。

In [ ]:
from pathlib import Path
import asyncio
import hashlib
import json
import sys
import time
from getpass import getpass
from urllib.parse import urlsplit, urlunsplit
from importlib.metadata import version

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "examples/flow_api.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 VLX-Flow 仓库内打开 Notebook。")
if str(PROJECT_ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "examples"))

import httpx
from websockets.asyncio.client import connect
from websockets.exceptions import ConnectionClosed
from IPython.display import display
from flow_api import RealtimeSession, api_root, verify_upload

# 保留已创建的对象，重复运行时不会丢失停止入口。
_api_sessions = globals().setdefault("_api_sessions", [])
for package in ("httpx", "websockets", "python-socks", "ipykernel"):
    print(f"{package}: {version(package)}")
print("环境就绪；尚未上传或连接模型。")

## 2 · 输入鉴权、视频与分析指令

令牌通过隐藏输入获取，不要写入源码。这里的 `INSTRUCTION` 是创建会话时的持续分析指令，后续追问使用另一个字段 `prompt`。

In [ ]:
if any(s.running or s.cleanup_blocked for s in _api_sessions):
    raise RuntimeError("请先运行停止单元格，关闭旧会话后再修改配置。")

API_ENDPOINT = api_root(input("平台 API 根地址（https://…）："))
API_KEY = getpass("访问令牌：").strip()
if not API_KEY:
    raise ValueError("访问令牌不能为空。")

VIDEO_PATH = PROJECT_ROOT / "examples/assets/videos/vlxflow-video01_keyframe_1s.mp4"
MODEL_ID = "VLX-Flow-1.5-10B"
INSTRUCTION = "持续观看视频画面, 并描述画面内容"
PARAMETERS = {
    "interval_seconds": 1000,  # 文档规定为毫秒，1000 即 1 秒
    "batch_size": 2,          # 每累计 2 张新帧触发分析；支持 1 或 2
    "window_size": 4,         # 窗口数，支持 1–8
    "temperature": 0.2,       # 0 <= 值 < 2
    "max_tokens": 32,         # 每轮 1–32 tokens
}
DURATION_SECONDS = 60  # 本地自动停止时限，不放入 API 请求
USE_PROXY = True       # 使用系统代理；如需直连改为 False
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(VIDEO_PATH.name)
print("视频：", VIDEO_PATH.relative_to(PROJECT_ROOT))
print("指令：", INSTRUCTION)
print("参数：", PARAMETERS)

## 3 · 上传本地视频

Realtime 使用服务端可访问的 URL。本单元格展示 multipart 上传的实际调用；只上传，不启动模型。相同文件在本次内核中可复用上传结果。

In [ ]:
if any(s.running or s.cleanup_blocked for s in _api_sessions):
    raise RuntimeError("请先关闭旧会话，再更换或上传视频。")
FORCE_UPLOAD = False
stat = VIDEO_PATH.stat()
upload_identity = (API_ENDPOINT, hashlib.sha256(API_KEY.encode()).hexdigest(),
                   str(VIDEO_PATH), stat.st_size, stat.st_mtime_ns)
upload_cache = globals().setdefault("_flow_upload_cache", {})
if not FORCE_UPLOAD and upload_identity in upload_cache:
    VIDEO_URL = upload_cache[upload_identity]
    print("复用当前内核的上传结果。")
else:
    VIDEO_URL = None
    async with httpx.AsyncClient(timeout=httpx.Timeout(300, connect=30), trust_env=USE_PROXY) as client:
        with VIDEO_PATH.open("rb") as video:
            response = await client.post(
                f"{API_ENDPOINT}/os/uploadFile",
                headers={"Authorization": f"Bearer {API_KEY}"},
                files={"file": (VIDEO_PATH.name, video, "video/mp4")},
                data={"type": "video/mp4"},
            )
    VIDEO_URL = verify_upload(response)  # 读取并校验 data.fileUrl
    upload_cache[upload_identity] = VIDEO_URL
    print("上传成功。")
UPLOADED_IDENTITY = upload_identity
print("URL 仅保存在内存中，尚未调用模型。")

## 4 · 理解创建会话的请求

顶层固定为 `type`、`request_id`、`data`；`source.type` 使用字符串 `"2"`。下方只预览，不发送。模型 ID 应以你的部署为准。

In [ ]:
create_preview = {
    "type": "session.create", "request_id": "每次创建生成唯一 ID",
    "data": {
        "model": MODEL_ID,
        "source": {"type": "2", "path": "<上传后得到的 VIDEO_URL>"},
        "instruction": INSTRUCTION,
        **PARAMETERS,
    },
}
print(json.dumps(create_preview, ensure_ascii=False, indent=2))

## 5 · 开始分析并接收流式输出

先运行下面两块函数定义，再运行「启动」块。连接地址、Bearer 鉴权、创建 JSON、`ws.recv()` 和关闭 JSON 都在下方代码中。

每个连接只有一个接收循环。`EventState`（在 `flow_api.py` 中）按 `response_id` 拼接 `response.text.delta`，以 `response.text.done` 的完整文本为准，用 `response.done` 判断终态；还负责去重、关联追问和累计用量。`algorithm.analysis` 是持续分析，`algorithm.inquiry` 是追问。

`RealtimeSession` 提供后台任务、参数检查和 REST 关闭兜底；这里把接收函数作为 `runner` 传入，**实际执行的接收代码就是本单元格**。

In [ ]:
import uuid

async def receive_session(session):
    session.started_at = time.monotonic()
    close_at = None
    parsed = urlsplit(session.endpoint)
    ws_url = urlunsplit(("wss", parsed.netloc, "/kernel-rs/v1/realtime/sessions", "", ""))
    try:
        async with connect(
            ws_url, additional_headers={"Authorization": f"Bearer {session.api_key}"},
            proxy=True if session.use_proxy else None,
            open_timeout=20, close_timeout=5, ping_interval=20, ping_timeout=20,
        ) as ws:
            session.ws = ws
            if session.stop_event.is_set():
                return
            create_request = {"type": "session.create", "request_id": "create-" + uuid.uuid4().hex, "data": {
                "model": session.remote_model,
                "source": {"type": "2", "path": session.video_url},
                "instruction": session.instruction_text,
                **session.parameters,
            }}
            await ws.send(json.dumps(create_request, ensure_ascii=False))
            session.status = "会话创建中"
            while True:
                session.elapsed = time.monotonic() - session.started_at
                if (session.stop_event.is_set() or session.elapsed >= session.duration_seconds) and close_at is None:
                    session.status = "正在关闭"
                    if not session.state.ready:
                        break
                    await ws.send(json.dumps({"type": "session.close", "request_id": uuid.uuid4().hex, "data": {}}))
                    close_at = time.monotonic()
                if close_at is not None and time.monotonic() - close_at >= 8:
                    break
                if close_at is None:
                    while not session.commands.empty():
                        await ws.send(json.dumps(session.commands.get_nowait(), ensure_ascii=False))
                session._publish(force=False)
                try:
                    raw = await asyncio.wait_for(ws.recv(), timeout=0.1)
                except asyncio.TimeoutError:
                    continue
                message = json.loads(raw)
                was_ready = session.state.ready
                session.state.consume(message)
                if session.state.ready and not was_ready:
                    session.playback_id += 1
                    session.status = "持续分析中"
                if session.state.fatal_error:
                    raise RuntimeError(session.state.fatal_error)
                if session.state.closed or session.state.cleanup_pending:
                    break
    except ConnectionClosed as exc:
        if not session.state.closed and close_at is None:
            session.failure = f"连接已断开（{type(exc).__name__}），正在检查会话关闭状态。"
    except asyncio.CancelledError:
        session.failure = "任务已取消，正在清理会话。"
    except Exception as exc:
        session.failure = f"{type(exc).__name__}: {exc}"
    finally:
        session.ws = None
        session.state.end_questions()
        session.elapsed = time.monotonic() - session.started_at
        try:
            session.cleanup_blocked = not await session._confirm_cleanup()
        except Exception as exc:
            session.cleanup_blocked = True
            session.state.notice = f"关闭确认失败：{exc}"
        if session.cleanup_blocked:
            session.status = "等待关闭确认"
        elif session.state.creation_rejected:
            session.status = "创建失败，未进入视频分析"
        elif session.state.closed:
            session.status = "会话已关闭"
        else:
            session.status = "连接已结束"
        session._publish()

In [ ]:
def show_session(session):
    # text/plain + display_id：只更新普通文本，不依赖 HTML、Widget 或前端文件。
    state = session.state
    lines = [f"[{session.label}] {session.status} · {session.elapsed:.1f} 秒"]
    row = state.last_analysis
    if row:
        lines.append(f"[持续分析 · {row['id']} · {row['status']}] {row['text']!s}")
    for row in list(state.inquiries.values())[-5:]:
        lines.append(f"[追问 · {row['id']} · {row['status']}] {row['question'] or '等待问题关联'}\n{row['text']}")
    for q in state.questions.values():
        if q.get('error'):
            lines.append("追问错误：" + session._safe_message(q['error']))
    if state.notice:
        lines.append("提示：" + session._safe_message(state.notice))
    if session.failure:
        lines.append("错误：" + session._safe_message(session.failure))
    lines.append(f"Token：{state.tokens if state.usage_known else '等待返回'}（包含追问）")
    session.text_output.update({'text/plain': '\n\n'.join(lines)}, raw=True)


def make_session(label, instruction):
    stat = VIDEO_PATH.stat()
    identity = (API_ENDPOINT, hashlib.sha256(API_KEY.encode()).hexdigest(),
                str(VIDEO_PATH), stat.st_size, stat.st_mtime_ns)
    if identity != UPLOADED_IDENTITY or not VIDEO_URL:
        raise RuntimeError("文件或鉴权已变化，请重新执行上传步骤。")
    session = RealtimeSession(
        endpoint=API_ENDPOINT, api_key=API_KEY, video_url=VIDEO_URL,
        model=MODEL_ID, instruction=instruction, duration_seconds=DURATION_SECONDS,
        use_proxy=USE_PROXY, runner=receive_session, **PARAMETERS,
    )
    session.label = label
    session.text_output = display({'text/plain': f'[{label}] 准备启动'}, raw=True, display_id=True)
    session.on_change = show_session
    _api_sessions.append(session)
    return session

### 启动

`start()` 通过 `asyncio.create_task` 启动上面定义的函数，然后立即返回，所以可以继续执行第 6 步。

文本增量更新在本块输出中；基础版原样显示最近一轮文本，包括 `...`。这不是模型延迟或检测效果保证。重复运行时会阻止重复创建活动会话。

In [ ]:
stat = VIDEO_PATH.stat()
identity = (API_ENDPOINT, hashlib.sha256(API_KEY.encode()).hexdigest(),
            str(VIDEO_PATH), stat.st_size, stat.st_mtime_ns)
if identity != UPLOADED_IDENTITY or not VIDEO_URL:
    raise RuntimeError("文件或鉴权已变化，请重新执行上传步骤。")

if any(s.running or s.cleanup_blocked for s in _api_sessions):
    raise RuntimeError("请先运行第 7 步停止旧会话。")
session = make_session("单会话", INSTRUCTION)
session.start()

## 6 · 在同一会话继续追问

等待首轮分析到达后，填入 `QUESTION` 并运行。每次修改问题后可再次运行；示例每路同一时间只允许一条未完成追问。

实际消息格式为：
```json
{"type": "response.create", "request_id": "ask-001", "data": {"prompt": "画面中有什么？"}}
```
追问不会停止持续分析，也不新建会话。结果可能先于 `command.result` 到达，因此按 `response_id` 关联，不按到达顺序猜测。问题针对服务端当前上下文，不能据此保证可查询任意历史帧。

In [ ]:
QUESTION = ""  # 填入问题后运行，例如："现在画面中的地形有什么特点？"
if QUESTION.strip():
    if not session.can_ask:
        raise RuntimeError("请等待首轮分析和上一问完成；已结束的会话需重新开始。")
    if len(QUESTION) > 2000:
        raise ValueError("追问最多 2000 字符。")
    request = {
        "type": "response.create", "request_id": uuid.uuid4().hex,
        "data": {"prompt": QUESTION},
    }
    session.state.register_question(request["request_id"], QUESTION)
    try:
        await session.ws.send(json.dumps(request, ensure_ascii=False))
    except Exception:
        session.state.questions[request["request_id"]]["status"] = "unknown"
        session._publish()
        raise RuntimeError("追问发送结果未确认，请先检查会话，勿自动重发。") from None
    print("问题已发送；回答将流式更新在启动单元格的文本输出中。")
else:
    print("修改 QUESTION 后运行本单元格。")

## 7 · 停止并确认结束

此块可重复运行。`shutdown()` 先通知接收循环发送 `session.close`，等关闭结果；有必要时使用 REST DELETE 兜底。关闭待确认时不要启动下一次。

`response.cancel` 只取消指定回答，不结束视频分析；本示例的停止使用 `session.close`。不要用重启内核代替正常停止。

In [ ]:
results = await asyncio.gather(*(s.shutdown() for s in _api_sessions), return_exceptions=True)
errors = [str(r) for r in results if isinstance(r, BaseException)]
for s in _api_sessions:
    print(f"[{s.label}] {s.status}；Token：{s.state.tokens if s.state.usage_known else '未返回'}")
if errors:
    raise RuntimeError("；".join(errors))
print("所有已启动任务均已结束，无待确认的关闭请求。")

## 8 · 可选：同一视频双路并行

前面已停止后再尝试。这里复用同一个 URL，创建两个 WebSocket 会话，分别使用描述和检测指令。需要平台允许至少两路并发；两路分别计量，处理进度可能不同。

把 `RUN_PARALLEL` 改为 `True` 才会启动。任一路报错会停止另一条活动会话；两路的状态与结果独立保存。检测指令只是示例，请根据视频内容修改，需实测触发行为。

In [ ]:
RUN_PARALLEL = False
DESCRIPTION = "持续观看视频画面, 并描述画面内容"
DETECTION = "持续观看视频画面, 当画面中出现夕阳的时候向我汇报"

if RUN_PARALLEL:
    if any(s.running or s.cleanup_blocked for s in _api_sessions):
        raise RuntimeError("请先运行第 7 步关闭已有会话。")
    parallel_sessions = {
        "describe": make_session("描述", DESCRIPTION),
        "detect": make_session("检测", DETECTION),
    }
    def show_parallel(changed):
        show_session(changed)
        if changed.failure or changed.state.fatal_error or changed.cleanup_blocked:
            for peer in parallel_sessions.values():
                if peer is not changed and peer.running and not peer.stop_event.is_set():
                    peer.stop()
    for s in parallel_sessions.values():
        s.on_change = show_parallel
        s.start()  # 两个后台任务独立接收，不要串行 await 两个长时接收循环
else:
    print("跳过并行；需要时改为 True 后运行本块。")

### 可选：向其中一路追问

明确选择目标，不能假设两路拥有相同的画面上下文。

In [ ]:
TARGET = "describe"  # describe 或 detect
PARALLEL_QUESTION = ""
if PARALLEL_QUESTION.strip():
    if TARGET not in globals().get("parallel_sessions", {}):
        raise RuntimeError("请先启动对应的并行会话。")
    selected = parallel_sessions[TARGET]
    if not selected.can_ask:
        raise RuntimeError("请等待目标会话首轮分析和上一问完成。")
    if len(PARALLEL_QUESTION) > 2000:
        raise ValueError("追问最多 2000 字符。")
    request = {"type": "response.create", "request_id": uuid.uuid4().hex,
               "data": {"prompt": PARALLEL_QUESTION}}
    selected.state.register_question(request["request_id"], PARALLEL_QUESTION)
    try:
        await selected.ws.send(json.dumps(request, ensure_ascii=False))
    except Exception:
        selected.state.questions[request["request_id"]]["status"] = "unknown"
        selected._publish()
        raise RuntimeError("追问发送结果未确认，请先检查会话，勿自动重发。") from None
else:
    print("需要时填写 PARALLEL_QUESTION 再运行。")

### 停止两路

完成并行试用后运行。也可以再次运行第 7 步统一停止。

In [ ]:
results = await asyncio.gather(*(s.shutdown() for s in globals().get("parallel_sessions", {}).values()), return_exceptions=True)
errors = [str(r) for r in results if isinstance(r, BaseException)]
if errors:
    raise RuntimeError("；".join(errors))
print("并行任务已结束；若未启动过则无需清理。")

提交前停止会话、清除输出后保存。令牌与临时视频 URL 不应写入仓库。此处仅为公开 API 客户端示例，不包含模型训练或推理实现。